In [5]:
# Install dependenciesimport numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy import stats

In [12]:
# LOAD DATA
data = pd.read_csv("/kaggle/input/default-of-credit-card-clients-dataset/UCI_Credit_Card.csv")

In [13]:
# Inspect dataset
print("Dataset shape:", data.shape)
print(data.head())

Dataset shape: (30000, 25)
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1    20000.0    2          2         1   24      2      2     -1     -1   
1   2   120000.0    2          2         2   26     -1      2      0      0   
2   3    90000.0    2          2         2   34      0      0      0      0   
3   4    50000.0    2          2         1   37      0      0      0      0   
4   5    50000.0    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...        0.0        0.0        0.0       0.0     689.0       0.0   
1  ...     3272.0     3455.0     3261.0       0.0    1000.0    1000.0   
2  ...    14331.0    14948.0    15549.0    1518.0    1500.0    1000.0   
3  ...    28314.0    28959.0    29547.0    2000.0    2019.0    1200.0   
4  ...    20940.0    19146.0    19131.0    2000.0   36681.0   10000.0   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default.payment.next.month

# INITIAL CLEANING

In [14]:
# Drop unnamed index column if present
if 'ID' in data.columns:
    data = data.drop(columns=['ID'])

In [17]:
# Drop columns with >50% missing values (none expected)
threshold = 0.5
data = data.loc[:, data.isnull().mean() < threshold]

In [19]:
# Drop rows with >50% missing values
data = data.loc[data.isnull().mean(axis=1) < threshold]

In [21]:
# Drop duplicates if any
data = data.drop_duplicates()

# HANDLE MISSING VALUES

In [23]:
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = data.select_dtypes(include=['object', 'category']).columns

num_imputer = SimpleImputer(strategy='median')
if len(num_cols) > 0:
    data[num_cols] = num_imputer.fit_transform(data[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    data[cat_cols] = cat_imputer.fit_transform(data[cat_cols])

# OUTLIER HANDLING (Z-score)

In [25]:
z_scores = np.abs(stats.zscore(data[num_cols]))
data = data[(z_scores < 3).all(axis=1)]

# ENCODING

In [26]:
# Example: 'EDUCATION', 'MARRIAGE' are categorical with integer levels
# Ensure they are treated as categorical
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE']
for col in categorical_features:
    data[col] = data[col].astype('category')

In [27]:
# One-hot encode low-cardinality features
data = pd.get_dummies(data, columns=categorical_features, drop_first=True)

# FEATURE SCALING

In [28]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data.drop(columns=['default.payment.next.month']))

# DIMENSIONALITY REDUCTION (PCA)

In [29]:
pca = PCA(n_components=0.95)  # Retain 95% variance
X_pca = pca.fit_transform(scaled_data)

# TRAIN-TEST SPLIT

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, 
    data['default.payment.next.month'], 
    test_size=0.2, 
    random_state=42, 
    stratify=data['default.payment.next.month']
)

# KNN MODEL TRAINING

In [31]:
k = 5
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train, y_train)

KNeighborsClassifier()

# EVALUATION

In [32]:
y_pred = knn.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

In [33]:
print("===============================================")
print(f"✅ KNN Model Accuracy: {acc:.4f}")
print("===============================================")
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

✅ KNN Model Accuracy: 0.7949
Confusion Matrix:
[[3793  301]
 [ 782  404]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.83      0.93      0.88      4094
         1.0       0.57      0.34      0.43      1186

    accuracy                           0.79      5280
   macro avg       0.70      0.63      0.65      5280
weighted avg       0.77      0.79      0.77      5280

